#  Project Insurance Cost Prediction

## 👤 Student Information

| Field           | Entry (fill in)                |
|-----------------|--------------------------------|
| **Name**        | _Ei Htoo Khaing, James Nguyen, Michelle_     |
| **Student #**   | _115400244,  ,_     |
| **Course/Section** |BDA500 NAA    |
| **Date**        | _2026-10-08_                   |


## Dataset: medical_cost_prediction_dataset.csv


## Step 1: Frame Both Questions

• Write one supervised question: What continuous value will the regression models predict?

• Explain the practical value of both analyses and identify likely risks or limitations.

__Supervised Question: How accurately can a patient's annual medical cost (USD) be predicted from demographics, lifestyle habits, and chronic health conditions?__


---


# Step 2: Load and Understand the Data
• Display head (), shape, info (), data types, summary statistics, and missing value counts.  
• Identify numeric, categorical, identifier, date, and potential leakage columns.  
• Document duplicate handling and any rows or columns removed, with reasons.  


In [24]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("/Users/ACT/Downloads/medical_cost_prediction_dataset.csv")

display(df.head())
print("Shape:", df.shape)
df.info()

,age,gender,bmi,smoker,diabetes,hypertension,heart_disease,asthma,physical_activity_level,daily_steps,sleep_hours,stress_level,doctor_visits_per_year,hospital_admissions,medication_count,insurance_type,insurance_coverage_pct,city_type,previous_year_cost,annual_medical_cost
0,69,Male,29.4,No,1,0,0,0,Medium,14825,4.4,8,1,0,4,Private,80,Semi-Urban,10885,2645.50
1,32,Female,22.9,No,1,0,0,0,Medium,3620,6.0,7,4,3,0,Government,64,Semi-Urban,18722,10959.70
2,89,Male,25.7,No,0,0,0,0,High,10578,4.5,7,2,0,3,NaN,0,Urban,4196,8409.80
3,78,Male,31.9,Yes,0,1,0,0,Low,6226,8.6,9,6,1,7,Government,70,Urban,11128,7996.62
4,38,Male,27.7,No,0,0,0,0,High,6253,5.7,3,6,0,6,Private,77,Urban,15110,3202.52


Shape: (5000, 20)
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      5000 non-null   int64  
 1   gender                   5000 non-null   str    
 2   bmi                      5000 non-null   float64
 3   smoker                   5000 non-null   str    
 4   diabetes                 5000 non-null   int64  
 5   hypertension             5000 non-null   int64  
 6   heart_disease            5000 non-null   int64  
 7   asthma                   5000 non-null   int64  
 8   physical_activity_level  5000 non-null   str    
 9   daily_steps              5000 non-null   int64  
 10  sleep_hours              5000 non-null   float64
 11  stress_level             5000 non-null   int64  
 12  doctor_visits_per_year   5000 non-null   int64  
 13  hospital_admissions      5000 non-null   int64  
 14  medication_count 

In [33]:
display(df.describe().T)                    # numeric
display(df.describe(include="str").T)    # categorical

print("Missing values per column:")
print(df.isna().sum())

,count,mean,std,min,25%,50%,75%,max
age,5000.0,53.299000,20.646851,18.00,36.0000,53.00,71.0000,89.0
bmi,5000.0,25.970820,5.046651,6.40,22.6000,25.90,29.4000,43.6
diabetes,5000.0,0.207600,0.405629,0.00,0.0000,0.00,0.0000,1.0
hypertension,5000.0,0.288000,0.452876,0.00,0.0000,0.00,1.0000,1.0
heart_disease,5000.0,0.142200,0.349290,0.00,0.0000,0.00,0.0000,1.0
asthma,5000.0,0.096400,0.295169,0.00,0.0000,0.00,0.0000,1.0
daily_steps,5000.0,7993.216800,4052.127069,1004.00,4545.0000,7989.00,11532.2500,14999.0
sleep_hours,5000.0,6.488140,1.443361,4.00,5.2000,6.50,7.7000,9.0
stress_level,5000.0,5.475400,2.892312,1.00,3.0000,5.00,8.0000,10.0
doctor_visits_per_year,5000.0,4.030600,2.010689,0.00,3.0000,4.00,5.0000,14.0


,count,unique,top,freq
gender,5000,2,Female,2519
smoker,5000,2,No,3779
physical_activity_level,5000,3,Low,1996
insurance_type,3952,2,Private,1979
city_type,5000,3,Urban,2474


Missing values per column:
age                           0
gender                        0
bmi                           0
smoker                        0
diabetes                      0
hypertension                  0
heart_disease                 0
asthma                        0
physical_activity_level       0
daily_steps                   0
sleep_hours                   0
stress_level                  0
doctor_visits_per_year        0
hospital_admissions           0
medication_count              0
insurance_type             1048
insurance_coverage_pct        0
city_type                     0
previous_year_cost            0
annual_medical_cost           0
dtype: int64


• Identify numeric, categorical, identifier, date, and potential leakage columns.  

In [34]:
target = ["annual_medical_cost"] # the target column

leakage_cols = [
    "previous_year_cost",       # last year's cost, copy of target column
    "doctor_visits_per_year",   # utilization happens alongside cost
    "hospital_admissions",
    "medication_count",
    "insurance_type",           # coverage directly reduces final cost
    "insurance_coverage_pct",
]

numeric_cols = [c for c in df.select_dtypes(include="number").columns if c not in leakage_cols + target] # identify numerical columns
categorical_cols = [c for c in df.select_dtypes(include="str").columns if c not in leakage_cols + target] # identify categorical columns

identifier_cols = []   # none in this dataset
date_cols = []         # none in this dataset

print("Target:", target)
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)
print("Identifier:", identifier_cols)
print("Date:", date_cols)
print("Potential leakage:", leakage_cols)

Target: ['annual_medical_cost']
Numeric: ['age', 'bmi', 'diabetes', 'hypertension', 'heart_disease', 'asthma', 'daily_steps', 'sleep_hours', 'stress_level']
Categorical: ['gender', 'smoker', 'physical_activity_level', 'city_type']
Identifier: []
Date: []
Potential leakage: ['previous_year_cost', 'doctor_visits_per_year', 'hospital_admissions', 'medication_count', 'insurance_type', 'insurance_coverage_pct']


• Document duplicate handling and any rows or columns removed, with reasons.  

In [ ]:
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)

df_clean = df.drop_duplicates().drop(columns=leakage_cols) # drop the duplicates as well as the potential leakage columns
print("Shape before:", df.shape)
print("Shape after:", df_clean.shape)

Duplicate rows: 0
Shape before: (5000, 20)
Shape after: (5000, 14)


### Data Cleaning Notes  
- **Duplicates:** Found 0 duplicate rows.  
- **Missing values:** None found, so no imputation needed.  
- **Identifier and date columns:** None present in this dataset.  
- **Columns removed (6):** previous_year_cost, doctor_visits_per_year, hospital_admissions, medication_count, insurance_type, insurance_coverage_pct.  
  - Reason: these are measured alongside or after healthcare costs happen, or directly change the final cost. Keeping them would leak target information and make the model look more accurate than it would be in real use.  
- **Final shape:** 5000 rows × 14 columns (13 features + target).  

## Step 3: Exploratory Data Analysis  
• Create histograms and boxplots for important numeric features.  
• Create a correlation heatmap for numeric variables.  
• Create at least two plots relating important predictors to the regression target.

## Step 4: Prepare Features  
• Handle missing values using a justified strategy.  
• Encode categorical variables using OneHotEncoder or justified ordinal encoding.  
• Scale numeric features using StandardScaler or justify MinMaxScaler..

## Step 5: Build Configuration  
• Use ColumnTransformer to combine numeric and categorical preprocessing.  
• Create train/test split, recommended test % given in the table below.  
• Fit regression preprocessing only on the training data, then transform validation or test data without refitting.  
• Use cross-validation numbers given in the table below on the training data for comparison and tuning.